# Severity-Aware Conformal Pipeline — end to end

Runs the whole study in one notebook on a single GPU (Google Colab **or** a local machine), one `(DATASET, GEN_MODEL)` pair at a time, then pools every run for the final analysis:
1. **Generate & score** — `GEN_MODEL` answers each question of `DATASET`, splits it into atomic claims, scores each with P(true). *(GPU)*
2. **Grade & tag severity** — an OpenAI judge labels each claim true/hallucination/unverifiable and tags a danger tier. *(OpenAI API)*
3. **Validate & calibrate** — Gate 2 AUROC, then global vs **severity-aware** CRC for the current run. *(CPU)*
4. **Pooled analysis** — pool each model's claims across **all** datasets and report the global-vs-severity-aware headline with cluster-bootstrap confidence intervals. *(CPU — no GPU or API needed)*

Caches are namespaced by **dataset and model** (`scored_claims__<dataset>__<model>.jsonl`, `graded_claims__<dataset>__<model>.jsonl`), so each pair writes its own result set and resumes independently. Parts 1–3 run one pair at a time; Part 4 reads **every** `graded_claims__*.jsonl` in `results/`, so once you have generated a few runs you can re-run Part 4 alone — locally, with no GPU or API — to rebuild the cross-dataset, cross-model tables.

**Datasets** (all share the K-QA `Must_have` / `Nice_to_have` statement format): `kqa` (K-QA) plus the non-K-QA MedLFQA subsets `liveqa`, `medicationqa`, `healthsearchqa`. To run another benchmark or model, change `DATASET` / `GEN_MODEL` and Run All.

### Before running
**On Google Colab** — Runtime → Change runtime type → **GPU**. Get the code with `!git clone <repo-url>` (then run from the repo root) or upload the `sac/` folder into the Files panel.

**Locally** — launch Jupyter from the repo root (so `sac/` imports) on a machine with a CUDA GPU for Parts 1–2. Part 4 (analysis) needs no GPU and runs anywhere.

**Credentials** — resolved at runtime, never written to disk; the setup cell reads each from an environment variable, then a Colab Secret, then prompts you:
- **OPENAI_API_KEY** — needed for Part 2 grading.
- **HF_TOKEN** — only for *gated* generators (e.g. Llama-3); click *Agree and access repository* on the model page first. Open models need no token — leave the prompt blank.

Then **Run All**. Sessions clear local files on disconnect, so download `results/` when done. To resume an interrupted run, put the saved `scored_claims__…` / `graded_claims__…` back into `results/` first — Parts 1–2 skip whatever is already there.

## Config + setup

In [ ]:
# ===================== CONFIG — edit, then Run All =====================
DATASET         = "kqa"       # which benchmark to run (one of DATASETS below)
GEN_MODEL       = "meta-llama/Meta-Llama-3-8B-Instruct"      # generator + scorer (GPU)
JUDGE_MODEL     = "gpt-4o"     # OpenAI grader + severity tagger
N_QUESTIONS     = None        # None = all questions; set an int to subsample
ALPHA_MARGINAL  = 0.10        # global risk budget
ALPHA_DANGEROUS = 0.05        # stricter budget for dangerous claims
ALPHA_BENIGN    = 0.15        # looser budget for benign claims
N_SPLITS        = 300         # random cal/test splits to average over
WORK            = "results"   # output folder for caches + tables
# ======================================================================

# Benchmarks, all in the same "Question + Must_have / Nice_to_have" statement
# format, so one loader and one pipeline handle them all. K-QA is from its own
# repo; the others are the non-K-QA subsets of MedLFQA (OLAPH) — combining K-QA
# with these gives independent datasets with no overlap.
DATASETS = {
    "kqa":            "https://raw.githubusercontent.com/Itaymanes/K-QA/main/dataset/questions_w_answers.jsonl",
    "liveqa":         "https://raw.githubusercontent.com/dmis-lab/OLAPH/main/MedLFQA/live_qa_test_MedLFQA.jsonl",
    "medicationqa":   "https://raw.githubusercontent.com/dmis-lab/OLAPH/main/MedLFQA/medication_qa_test_MedLFQA.jsonl",
    "healthsearchqa": "https://raw.githubusercontent.com/dmis-lab/OLAPH/main/MedLFQA/healthsearch_qa_test_MedLFQA.jsonl",
}

# Filesystem-safe tag from the model name. Outputs are namespaced by BOTH dataset
# and model, so every (dataset, model) run writes and resumes its own cache and
# never collides; the final cell aggregates every run it finds.
MODEL_SLUG      = GEN_MODEL.split("/")[-1].lower().replace(".", "-")

### Decomposition prompt — pick the one that matches your model

The next cell defines two prompts that split each answer into atomic claims:

- **`DECOMPOSE_PROMPT_GENERAL`** — for general instruction-tuned models (Llama-3-Instruct, Mistral-7B-Instruct). Used for our main results.
- **`DECOMPOSE_PROMPT_MEDICAL`** — a stronger, few-shot prompt for medically fine-tuned models (BioMistral-7B, Llama3-OpenBioLLM-8B), which otherwise dump the whole answer as one block.

Set `DECOMPOSE_PROMPT` to the one matching `GEN_MODEL`. If a run produces only ~1 (long) claim per question, you picked the wrong one — switch to the medical prompt and re-run Part 1.

In [ ]:
# ===== Decomposition prompt — set DECOMPOSE_PROMPT to match GEN_MODEL =====
# General instruction-tuned models (Llama-3-Instruct, Mistral-7B-Instruct, ...)
# reliably emit one claim per line from a short instruction. This is the prompt
# used for our main results — keep it for those models.
DECOMPOSE_PROMPT_GENERAL = """You are decomposing a medical answer into atomic claims.
An atomic claim is a single, self-contained, verifiable statement of fact.
Output ONE claim per line, no numbering, no commentary.

Question: {question}
Answer: {answer}

Atomic claims:"""

# Medically fine-tuned models (e.g. BioMistral-7B, Llama3-OpenBioLLM-8B) tend to
# ignore a terse instruction and dump the whole answer as one block. This version
# is explicit about the format and includes a worked example (few-shot) to force
# one claim per line. Switch to it whenever a model produces ~1 long "claim" per
# question (check Part 1's claims-per-question count).
DECOMPOSE_PROMPT_MEDICAL = """You are a careful medical fact-checker. Break the ANSWER into atomic claims.
An atomic claim is a single, self-contained, verifiable statement of exactly one fact.

Output format (follow exactly):
- Write exactly ONE claim per line.
- Put a line break between every claim.
- Never put two claims on one line; never join them with "1.", "2.", "and", or ";".
- Output only the claims: no numbering, no bullets, no headings, no extra text.

Example
Question: What is amoxicillin?
Answer: Amoxicillin is a penicillin antibiotic. It treats bacterial infections, and it does not work against viruses.
Atomic claims:
Amoxicillin is a penicillin antibiotic.
Amoxicillin treats bacterial infections.
Amoxicillin does not work against viruses.

Now decompose the following answer in the same way.
Question: {question}
Answer: {answer}
Atomic claims:"""

# Active prompt — use DECOMPOSE_PROMPT_MEDICAL for medically fine-tuned models.
DECOMPOSE_PROMPT = DECOMPOSE_PROMPT_GENERAL

In [ ]:
# Pin to ONE GPU before importing torch. A multi-GPU box would otherwise try to
# shard the 4-bit generator across every device and fail to load; one GPU runs
# the whole pipeline. No-op on a single-GPU machine (Colab, most workstations).
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

!pip -q install -U transformers bitsandbytes accelerate openai

import sys, glob, getpass

# Find the sac/ package whether you cloned the repo and run from its root, or
# uploaded the sac/ folder (Colab Files panel).
hits = glob.glob("sac/__init__.py") or glob.glob("**/sac/__init__.py", recursive=True)
if not hits:
    raise FileNotFoundError("sac/ not found - clone the repo and run from its root, "
                            "or upload the sac/ folder first.")
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(hits[0]))))

os.makedirs(WORK, exist_ok=True)
DATA   = f"{WORK}/{DATASET}.jsonl"                               # downloaded benchmark
SCORED = f"{WORK}/scored_claims__{DATASET}__{MODEL_SLUG}.jsonl"  # Part 1 output: generated + P(true)
GRADED = f"{WORK}/graded_claims__{DATASET}__{MODEL_SLUG}.jsonl"  # Part 2 output: + labels + severity

# Download the chosen benchmark (Question + Must_have / Nice_to_have statements).
DATA_URL = DATASETS[DATASET]
!wget -q -O {DATA} {DATA_URL}


def get_credential(name, prompt, required=True):
    """Resolve an API key without hardcoding it, in this order:
        1. environment variable (export it yourself for unattended runs)
        2. Colab Secret        (left sidebar -> key icon)
        3. interactive prompt  (typed in, kept only in memory)
    """
    if os.environ.get(name):
        return os.environ[name]
    try:                                            # Colab Secrets
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return getpass.getpass(prompt)                  # interactive fallback


# OPENAI_API_KEY is always needed (Part 2 grading). HF_TOKEN is only for gated
# generators (e.g. Llama-3) - those also need a one-time "Agree and access
# repository" click on the model's Hugging Face page. Open models need no token.
os.environ["OPENAI_API_KEY"] = get_credential("OPENAI_API_KEY", "OpenAI API key: ")
hf_token = get_credential("HF_TOKEN", "Hugging Face token (blank for open models): ",
                          required=False).strip()
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)

import numpy as np
from collections import Counter
from tqdm.auto import tqdm
from sac.kqa_loader import load_kqa, gold_statements, load_physician_nli   # shared statement format
from sac.hf_backend import HFBackend
from sac.decompose import decompose
from sac.scoring import score_claim
from sac.grader import grade_claim, tag_severity
from sac.cache import append_claims, existing_claim_ids, load_claims
from sac.crc import (Claim, crc_calibrate, crc_calibrate_stratified, apply_global,
                     apply_stratified, retention, realized_risk_marginal, tier_risk_marginal)
from sac.validate import score_auroc, gate2_pass, unverifiable_count, labeler_agreement
from sac.openai_judge import OpenAIJudge

# Resume: Parts 1-2 reuse whatever is already in WORK (keyed by claim id), so an
# interrupted run continues where it stopped - just keep the existing results/ folder.
items = load_kqa(DATA)[:N_QUESTIONS]
print(f"{DATASET}: {len(items)} questions loaded")

## Part 1 — Generate answers & score claims  (GPU, checkpointed → `scored_claims__<dataset>__<model>.jsonl`)

In [ ]:
backend = HFBackend(model_id=GEN_MODEL)

done = existing_claim_ids(SCORED)
todo = [it for it in items if not any(cid.startswith(it.qid + "_") for cid in done)]
print(f"{len(items) - len(todo)} questions already done, generating {len(todo)} more")
for it in tqdm(todo, desc="generate"):
    answer = backend.generate(f"Question: {it.question}\nAnswer:")
    claims = [Claim(text=ct, confidence=score_claim(ct, backend),
                    answer_id=it.qid, claim_id=f"{it.qid}_c{j}")
              for j, ct in enumerate(decompose(it.question, answer, backend, prompt_template=DECOMPOSE_PROMPT))]
    append_claims(SCORED, claims)          # checkpoint after each question
# sanity check: ~5-25 claims/question is healthy; ~1 means decomposition failed
sc = load_claims(SCORED)
print(f"scored_claims: {len(sc)} claims over {len(set(c.answer_id for c in sc))} questions "
      f"({len(sc)/max(1,len(set(c.answer_id for c in sc))):.1f} per question)")

## Part 2 — Grade & tag severity  (OpenAI, checkpointed → `graded_claims__<dataset>__<model>.jsonl`)

In [ ]:
items_by_id = {it.qid: it for it in items}
judge = OpenAIJudge(model=JUDGE_MODEL)

done    = existing_claim_ids(GRADED)
pending = [c for c in load_claims(SCORED) if c.claim_id not in done]
for c in tqdm(pending, desc="grade"):
    statements = gold_statements(items_by_id[c.answer_id])
    c.label, c.grader_rationale    = grade_claim(c.text, statements, judge)
    c.tier,  c.severity_rationale  = tag_severity(c.text, judge)
    append_claims(GRADED, [c])             # checkpoint after each claim

graded = load_claims(GRADED)
print("labels:", Counter(c.label for c in graded))   # mix of 0 / 1 / -1
print("tiers :", Counter(c.tier  for c in graded))   # dangerous / benign

## Part 3 — Validate & calibrate  (CPU)

### Gate 2 — does the score separate truth from hallucination? (AUROC ≥ 0.70)

In [ ]:
graded     = load_claims(GRADED)
verifiable = [c for c in graded if c.label in (0, 1)]   # CRC ignores unverifiable (-1)
au = score_auroc(graded)
print(f"GATE 2  AUROC = {au:.3f}  ->  {'PASS' if gate2_pass(au) else 'FAIL (<0.70)'}")
print(f"{len(verifiable)} verifiable | {unverifiable_count(graded)} unverifiable (excluded)")

### Headline — risk & retention averaged over many splits

Global CRC controls only the *marginal* (overall) risk; severity-aware CRC additionally holds the **dangerous** tier to a stricter budget. Averaged over `N_SPLITS` random calibration/test splits so a single unlucky split can't mislead.

In [ ]:
def tier_retention(claims, kept, tier):
    kept = np.asarray(kept); idx = np.array([c.tier == tier for c in claims])
    lab = np.array([c.label for c in claims]); nt = np.sum(idx & (lab == 0))
    return np.sum(kept & idx & (lab == 0)) / nt if nt else 0.0

G = {k: [] for k in ("d_risk", "b_risk", "d_ret", "b_ret", "marg")}
S = {k: [] for k in ("d_risk", "b_risk", "d_ret", "b_ret")}
for seed in range(N_SPLITS):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(verifiable)); k = len(verifiable) // 2
    cal = [verifiable[i] for i in idx[:k]]
    te  = [verifiable[i] for i in idx[k:]]

    lam = crc_calibrate([c.confidence for c in cal], [c.label for c in cal], ALPHA_MARGINAL)
    kg = apply_global(te, lam)
    G["marg"].append(realized_risk_marginal(te, kg))
    G["d_risk"].append(tier_risk_marginal(te, kg, "dangerous")); G["b_risk"].append(tier_risk_marginal(te, kg, "benign"))
    G["d_ret"].append(tier_retention(te, kg, "dangerous"));      G["b_ret"].append(tier_retention(te, kg, "benign"))

    th = crc_calibrate_stratified(cal, {"dangerous": ALPHA_DANGEROUS, "benign": ALPHA_BENIGN})
    ks = apply_stratified(te, th)
    S["d_risk"].append(tier_risk_marginal(te, ks, "dangerous")); S["b_risk"].append(tier_risk_marginal(te, ks, "benign"))
    S["d_ret"].append(tier_retention(te, ks, "dangerous"));      S["b_ret"].append(tier_retention(te, ks, "benign"))

m = np.mean
print(f"Averaged over {N_SPLITS} splits  "
      f"(alpha: marginal={ALPHA_MARGINAL}, dangerous={ALPHA_DANGEROUS}, benign={ALPHA_BENIGN})\n")
print(f"                      GLOBAL    SEVERITY-AWARE")
print(f"  dangerous risk       {m(G['d_risk']):.3f}      {m(S['d_risk']):.3f}     (target {ALPHA_DANGEROUS})")
print(f"  benign    risk       {m(G['b_risk']):.3f}      {m(S['b_risk']):.3f}     (target {ALPHA_BENIGN})")
print(f"  dangerous retention  {m(G['d_ret']):.3f}      {m(S['d_ret']):.3f}")
print(f"  benign    retention  {m(G['b_ret']):.3f}      {m(S['b_ret']):.3f}")
print(f"  marginal  risk       {m(G['marg']):.3f}      --        (target {ALPHA_MARGINAL})")
print(f"\n  P(dangerous risk > {ALPHA_DANGEROUS}):  global={m(np.array(G['d_risk'])>ALPHA_DANGEROUS):.2f}   "
      f"severity-aware={m(np.array(S['d_risk'])>ALPHA_DANGEROUS):.2f}")
print("  ^ how often each method violates the danger budget.")

### Generalization — does the guarantee hold across datasets and models?

Run Parts 1–2 once per `(DATASET, GEN_MODEL)` pair (change the config, Run All); each run writes its own `graded_claims__<dataset>__<model>.jsonl` into `results/`. This cell loads **every** such file present and prints the global-vs-severity-aware headline for each. To add a dataset or model, run the pipeline again with the new config (or drop another `graded_claims__*.jsonl` into `results/`) and re-run this cell — no code edits.

The claim being tested: severity-aware risk control (`dRisk-S`) stays within the dangerous budget for *every* dataset and model, while retention (`dRet-S`) scales with score quality.

In [ ]:
# Multi-dataset / multi-model generalization table.
#
# What it does: finds every per-run graded file in WORK (one per dataset x model
# you have run) and re-runs the global-vs-severity-aware headline for each, using
# the same averaged-over-N_SPLITS procedure as the cell above (sac.ablation.run_ablation).
#
# Output: one row per (dataset, model) with
#   AUROC    discrimination of the P(true) score
#   dRisk-G  dangerous-tier risk under a single global threshold
#   dRisk-S  dangerous-tier risk under severity-aware CRC  (should be <= ALPHA_DANGEROUS)
#   dRet-G   fraction of true dangerous claims kept, global
#   dRet-S   fraction of true dangerous claims kept, severity-aware
#   P>a-G    how often the global method exceeds the danger budget across splits
#   P>a-S    how often severity-aware exceeds it
#
# Reading: dRisk-S stays within budget for every dataset and model (validity is
# universal); dRet-S rises with AUROC (a better score keeps more true claims).
from sac.ablation import run_ablation

run_files = sorted(glob.glob(f"{WORK}/graded_claims__*.jsonl"))
print(f"{len(run_files)} (dataset, model) run(s) found\n")

header = f"{'dataset':15s} {'model':24s} AUROC | dRisk-G dRisk-S | dRet-G dRet-S | P>a-G P>a-S"
print(header)
print("-" * len(header))
for path in run_files:
    tag = os.path.basename(path)[len("graded_claims__"):-len(".jsonl")]
    dataset, _, model = tag.partition("__")            # filename is dataset__model
    claims = [c for c in load_claims(path) if c.label in (0, 1)]   # verifiable only
    r = run_ablation(claims, ALPHA_MARGINAL, ALPHA_DANGEROUS, ALPHA_BENIGN, N_SPLITS)
    print(f"{dataset:15s} {model:24s} {r['auroc']:.3f} | {r['g_d_risk']:7.3f} {r['s_d_risk']:7.3f} | "
          f"{r['g_d_ret']:6.3f} {r['s_d_ret']:6.3f} | {r['g_violation']:5.2f} {r['s_violation']:5.2f}")

print(f"\nValidity: dRisk-S should stay <= {ALPHA_DANGEROUS} for every (dataset, model).")
print( "Quality : dRet-S (true dangerous claims kept) tracks AUROC.")

## Part 4 — Pooled analysis across datasets, with confidence intervals

Part 3 reports each `(dataset, model)` run on its own. Here we **pool** every dataset for a given model into one claim set (confidence = that model's P(true), comparable within a model) and report the global-vs-severity-aware headline on the pool — the paper's main result. Pooling raises the count of *dangerous hallucinations* (the rare events that drive the uncertainty), so the estimates tighten and we can attach honest 95% confidence intervals.

The interval is a **cluster bootstrap by question**: claims from the same answer are correlated, so we resample whole *questions* with replacement (not individual claims) and recompute the dangerous-tier risk many times. A naive per-claim bootstrap would understate the uncertainty.

> Part 4 reads only `results/graded_claims__*.jsonl`, so it runs standalone — locally, no GPU or API. Just run the **Config** cell, then the two cells below.

In [ ]:
# ===== Part 4a - pool each model across datasets; point-estimate table =====
import os, glob
import numpy as np
from sac.cache import load_claims
from sac.ablation import run_ablation

ALPHAS = dict(alpha_marginal=ALPHA_MARGINAL, alpha_dangerous=ALPHA_DANGEROUS, alpha_benign=ALPHA_BENIGN)
DATASET_ORDER = ["kqa", "liveqa", "medicationqa", "healthsearchqa"]

def qid_of(c):                       # question id = claim_id minus its trailing _<n>
    return c.claim_id.rsplit("_", 1)[0]

def n_dang_halluc(cs):
    return sum(1 for c in cs if c.label == 1 and c.tier == "dangerous")

# {model: {dataset: [verifiable claims]}}  from every graded file present
grid = {}
for path in sorted(glob.glob(f"{WORK}/graded_claims__*.jsonl")):
    tag = os.path.basename(path)[len("graded_claims__"):-len(".jsonl")]
    ds, _, model = tag.partition("__")
    grid.setdefault(model, {})[ds] = [c for c in load_claims(path) if c.label in (0, 1)]

POOLS = {}                           # stash pools for the bootstrap cell below
for model in sorted(grid):
    per_ds = grid[model]
    pool = [c for ds in DATASET_ORDER if ds in per_ds for c in per_ds[ds]]
    POOLS[model] = pool
    nq = len({qid_of(c) for c in pool})
    print("=" * 80)
    print(f"{model}   {len(pool)} verifiable claims | {nq} questions | "
          f"{n_dang_halluc(pool)} dangerous hallucinations")
    print("=" * 80)
    hdr = (f"  {'dataset':14s}{'AUROC':>7}{'dRisk-G':>9}{'dRisk-S':>9}"
           f"{'dRet-G':>8}{'dRet-S':>8}{'P>a-G':>7}{'P>a-S':>7}{'dHall':>7}")
    print(hdr); print("  " + "-" * (len(hdr) - 2))
    for ds in [d for d in DATASET_ORDER if d in per_ds] + ["POOLED"]:
        cs = pool if ds == "POOLED" else per_ds[ds]
        r = run_ablation(cs, **ALPHAS, n_splits=N_SPLITS)
        print(f"  {ds:14s}{r['auroc']:>7.3f}{r['g_d_risk']:>9.3f}{r['s_d_risk']:>9.3f}"
              f"{r['g_d_ret']:>8.3f}{r['s_d_ret']:>8.3f}"
              f"{r['g_violation']:>7.2f}{r['s_violation']:>7.2f}{n_dang_halluc(cs):>7}")
    print()
print(f"dRisk-S should sit at/under the dangerous budget {ALPHA_DANGEROUS}; dRisk-G busts it.")

In [ ]:
# ===== Part 4b - cluster-bootstrap-by-question 95% CI on dangerous risk =====
# Resample whole QUESTIONS with replacement, recompute the dangerous-tier risk,
# repeat n_boot times; the middle 95% of the values is the CI. ~1-2 min per model.
def bootstrap_dangerous_ci(claims, n_boot=400, n_splits=25, seed=0):
    by_q = {}
    for c in claims:
        by_q.setdefault(qid_of(c), []).append(c)
    qids = list(by_q); rng = np.random.default_rng(seed)
    g, s = [], []
    for _ in range(n_boot):
        pick = rng.choice(len(qids), size=len(qids), replace=True)
        boot = [c for i in pick for c in by_q[qids[i]]]
        r = run_ablation(boot, **ALPHAS, n_splits=n_splits)
        g.append(r["g_d_risk"]); s.append(r["s_d_risk"])
    pct = lambda a: (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
    return pct(g), pct(s)

for model, pool in POOLS.items():
    (g_lo, g_hi), (s_lo, s_hi) = bootstrap_dangerous_ci(pool)
    print(f"{model}  (pooled, {n_dang_halluc(pool)} dangerous hallucinations)")
    print(f"   global         dangerous risk: 95% CI [{g_lo:.3f}, {g_hi:.3f}]")
    print(f"   severity-aware dangerous risk: 95% CI [{s_lo:.3f}, {s_hi:.3f}]   budget {ALPHA_DANGEROUS}")
    print()

### (Optional) Gate 1 — does our judge agree with physicians?

Validates the OpenAI grader against K-QA's physician NLI annotations. One API call per pair (subsampled). Skip unless you want the agreement number for the paper.

In [ ]:
N_NLI = 200   # subsample of physician-annotated pairs (None = all ~399)

!wget -q -O {WORK}/nli.csv https://raw.githubusercontent.com/Itaymanes/K-QA/main/dataset/NLI_medical_annotator.csv
pairs = load_physician_nli(f"{WORK}/nli.csv")
if N_NLI:
    pairs = pairs[:N_NLI]
g1 = labeler_agreement(pairs, OpenAIJudge(model=JUDGE_MODEL))
print(f"GATE 1  labeler accuracy={g1['accuracy']:.3f}  kappa={g1['kappa']:.3f}  n={g1['n']}")